# Querying Data with the SQLAlchemy ORM

Once you have a database engine and model classes, you interact with the database through a **Session**. The SQLAlchemy Core API provides `select()`, `update()`, and `delete()` functions that feel like writing SQL but work with your Python model objects.

## Setup: Engine, Base, and Models

We'll reuse the `CalendarEvent` model from the data modeling module.

In [1]:
from datetime import date
from sqlalchemy import create_engine, Integer, String, Date, Boolean
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

engine = create_engine("sqlite:///countdown.db")

class Base(DeclarativeBase):
    pass

class Calendar(Base):
    __tablename__ = "calendar"
    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    name: Mapped[str] = mapped_column(String(100), nullable=False)

class CalendarEvent(Base):
    __tablename__ = "calendar_event"
    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    event_name: Mapped[str] = mapped_column(String(100), nullable=False)
    event_date: Mapped[date] = mapped_column(Date, default=date.today)
    priority: Mapped[int] = mapped_column(Integer, default=5)
    is_private: Mapped[bool] = mapped_column(Boolean, default=False)

Base.metadata.create_all(bind=engine)
print("Ready")

Ready


## Sessions

A **Session** is how you interact with the database in SQLAlchemy:
- Queries are sent and results returned through sessions
- A session is bound to a SQLAlchemy engine

Use a `with` block to automatically close the session.

In [2]:
from sqlalchemy.orm import Session

# Add an object using a session
with Session(bind=engine) as sess:
    calendar = Calendar(name="My Calendar")
    sess.add(calendar)
    sess.commit()

print("Calendar added")

Calendar added


## Retrieving All Objects

Use `select(ModelClass)` to build a query, then `sess.scalars(query).all()` to get a list of ORM objects.

In [3]:
from sqlalchemy import select
from sqlalchemy.orm import Session

with Session(bind=engine) as sess:
    query = select(Calendar)
    calendars = sess.scalars(query).all()

for cal in calendars:
    print(f"id={cal.id}  name={cal.name}")

id=1  name=My Calendar


## Retrieving a Single Object

Chain `.where()` to filter, then use `.first()` instead of `.all()` to get one item (or `None`).

In [4]:
from sqlalchemy import select
from sqlalchemy.orm import Session

calendar_id = 1

with Session(bind=engine) as sess:
    query = select(Calendar).where(Calendar.id == calendar_id)
    calendar = sess.scalars(query).first()

if calendar:
    print(f"Found: {calendar.name}")
else:
    print("Not found")

Found: My Calendar


## Searching by Name

Use `.contains()` on a string column for a case-insensitive substring search.

In [5]:
from sqlalchemy import select
from sqlalchemy.orm import Session

search_term = "My"

with Session(bind=engine) as sess:
    query = select(Calendar).where(Calendar.name.contains(search_term))
    results = sess.scalars(query).all()

for cal in results:
    print(cal.name)

My Calendar


## Updating Objects

Use `update(ModelClass).where(...).values(...)` to build an UPDATE statement, then `sess.execute(query)` and `sess.commit()`. Check `result.rowcount` to confirm the update succeeded.

In [6]:
from sqlalchemy import update
from sqlalchemy.orm import Session

calendar_id = 1
new_name = "Personal Calendar"

with Session(bind=engine) as sess:
    query = (
        update(Calendar)
        .where(Calendar.id == calendar_id)
        .values(name=new_name)
    )
    result = sess.execute(query)
    sess.commit()

    if result.rowcount > 0:
        print("Update succeeded")
    else:
        print("No rows updated")

Update succeeded


## Deleting Objects

Use `delete(ModelClass).where(...)` to build a DELETE statement, then execute and commit.

In [7]:
from sqlalchemy import delete
from sqlalchemy.orm import Session

calendar_id = 1

with Session(bind=engine) as sess:
    query = delete(Calendar).where(Calendar.id == calendar_id)
    result = sess.execute(query)
    sess.commit()

    if result.rowcount > 0:
        print("Delete succeeded")
    else:
        print("Nothing to delete")

Delete succeeded


## Summary: Querying SQLAlchemy

| Operation | Import | Example |
|-----------|--------|---------|
| SELECT all | `select` | `select(Model)` |
| SELECT where | `select` | `select(Model).where(Model.col == val)` |
| SELECT one | `select` | `...scalars(q).first()` |
| UPDATE | `update` | `update(Model).where(...).values(...)` |
| DELETE | `delete` | `delete(Model).where(...)` |

Always call `sess.commit()` after UPDATE and DELETE.